# Compute Forward Operator

## 📦 Imports

In [1]:
import mindmap as mm 
import mne
import pyvista as pv

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
- `bem`: Boundary Element Model (Head conductor model).
- `src-raw`: Source space (before computing forward).
- `eeg`: EEG.
- `trans`: Affine transformation for EEG coregistration.

🎚️ **<font color=pink>Configuration</font>**:
- `fconfig`: Configuration file.
- `sub`: Subject id (position in list SUBS)
- `hmodel`: Type of head model ["bem", "bem-sim"]. Whether to compute the forward solution for inverse problems or simulations.

In [2]:
# ------------- Parameters ---------------
fconfig = "../../data/config.yml"
sub     = "intra-12"
hmodel  = "bem"
task    = "spk"
# ----------------------------------------

reader = mm.io.TutoReader(fconfig)
mm_dir = reader.config["MM_DIR"]

# data
data = reader(sub, hmodel, "src-raw", "ave", "trans", task=task)

/home/nino/Code/mindmap/data/derivatives/mindmap/sub-intra-12/src/sub-intra-12_src.fif
DATA SUMMARY
Dataset: tuto
Subject: intra-12
Task: spk
Run: None
Retrieved objects:
- ave
- trans
- bem
- src-raw
Missing objects:


In [ ]:
data["src-raw"][0]["pinfo"]

TypeError: list indices must be integers or slices, not tuple

📊 **Check coregistration**

In [9]:
head = data[hmodel]["surfs"][0]
sens = data["ave"]._get_channel_positions()
sens = mne.transforms.apply_trans(data["trans"], sens)

# plot
head = mm.head.surf_to_polydata(head["rr"], head["tris"])
sens = pv.PolyData(sens)

p = pv.Plotter()
p.add_mesh(head, color="#D2A18C", smooth_shading=False, label="head")
p.add_mesh(sens, render_points_as_spheres=True, color="black", label="sensors", point_size=8)

p.add_legend(face="circle")
p.show()

Widget(value='<iframe src="http://localhost:33639/index.html?ui=P_0x7140a9e89fd0_0&reconnect=auto" class="pyvi…

## ⚙️ Compute Forward Solution

In [10]:
forward = mne.make_forward_solution(
    info    = data["ave"].info,
    trans   = data["trans"],
    src     = data["src-raw"],
    bem     = data[hmodel],
    meg     = False,
    eeg     = True,
    verbose = "warning",
)

In [11]:
if reader.config.get("TASKS"):
    print("yes")
else:
    print("no")

yes


📊 **Plot leadfields**

In [12]:
mm.viz.plot_leadfield(data["ave"], forward).show()

    Average patch normals will be employed in the rotation to the local surface coordinates....
    Converting to surface-based source orientations...
    [done]


Widget(value='<iframe src="http://localhost:33639/index.html?ui=P_0x7140ab201490_1&reconnect=auto" class="pyvi…

💾 **Save Forward Solution** <br>
🎚️ **<font color=pink>Save</font>**:
- `overwrite`: Overwrite existing forward solution.


In [ ]:
# ------------ Parameters -------------
overwrite = True

file = mm.io.get_bidspath(
    root        = reader.config["MM_DIR"],
    subject     = sub,
    datatype    = "fwd",
    task        = None,
    session     = None,
    description = None,
)
# -------------------------------------

file.parent.mkdir(parents=True, exist_ok=True)

mne.write_forward_solution(file, forward, overwrite=overwrite, verbose="warning")
print(f"\033[92mSuccessfully saved forward solution for subject {data['sub']}\033[0m")

Successfully saved forward solution for subject intra-12
